In [1]:
from dotenv import load_dotenv

load_dotenv()

True

## Summarize messages

In [5]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents.middleware import SummarizationMiddleware

agent = create_agent(
    model="google_genai:gemini-3.5-flash-lite",
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model="google_genai:gemini-3.5-flash-lite",
            trigger=("tokens", 100),
            keep=("messages", 1)
        )
    ],
)

In [6]:
from langchain.messages import HumanMessage, AIMessage
from pprint import pprint

response = agent.invoke(
    {"messages": [
        HumanMessage(content="What is the capital of the moon?"),
        AIMessage(content="The capital of the moon is Lunapolis."),
        HumanMessage(content="What is the weather in Lunapolis?"),
        AIMessage(content="Skies are clear, with a high of 120C and a low of -100C."),
        HumanMessage(content="How many cheese miners live in Lunapolis?"),
        AIMessage(content="There are 100,000 cheese miners living in Lunapolis."),
        HumanMessage(content="Do you think the cheese miners' union will strike?"),
        AIMessage(content="Yes, because they are unhappy with the new president."),
        HumanMessage(content="If you were Lunapolis' new president how would you respond to the cheese miners' union?"),
        ]},
    {"configurable": {"thread_id": "1"}}
)

pprint(response)

{'messages': [HumanMessage(content='Here is a summary of the conversation to date:\n\n## SESSION INTENT\n\nThe user is asking fictional/creative questions about the moon\'s fictional capital, Lunapolis, including its capital, weather, population of cheese miners, and political/labor situation.\n\n## SUMMARY\n\n- Established the fictional capital of the moon as "Lunapolis".\n- Noted current fictional weather in Lunapolis: clear skies, high of 120°C, low of -100°C.\n- Noted a fictional population of 100,000 cheese miners living in Lunapolis.\n- Discussed the likelihood of a cheese miners\' union strike due to unhappiness with the new president.\n- No specific constraints, rejected options, or key architectural decisions to report.\n\n## ARTIFACTS\n\nNone\n\n## NEXT STEPS\n\n- Answer any follow-up questions the user may have regarding Lunapolis, its cheese miners, or related fictional topics.', additional_kwargs={'lc_source': 'summarization'}, response_metadata={}, id='45c97f23-964d-4f51-

In [9]:
print(response["messages"][0].content)

Here is a summary of the conversation to date:

## SESSION INTENT

The user is asking fictional/creative questions about the moon's fictional capital, Lunapolis, including its capital, weather, population of cheese miners, and political/labor situation.

## SUMMARY

- Established the fictional capital of the moon as "Lunapolis".
- Noted current fictional weather in Lunapolis: clear skies, high of 120°C, low of -100°C.
- Noted a fictional population of 100,000 cheese miners living in Lunapolis.
- Discussed the likelihood of a cheese miners' union strike due to unhappiness with the new president.
- No specific constraints, rejected options, or key architectural decisions to report.

## ARTIFACTS

None

## NEXT STEPS

- Answer any follow-up questions the user may have regarding Lunapolis, its cheese miners, or related fictional topics.


## Trim/delete messages

In [10]:
from typing import Any
from langchain.agents import AgentState
from langchain.messages import RemoveMessage
from langgraph.runtime import Runtime
from langchain.agents.middleware import before_agent
from langchain.messages import ToolMessage

@before_agent
def trim_messages(state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
    """Remove all the tool messages from the state"""
    messages = state["messages"]

    tool_messages = [m for m in messages if isinstance(m, ToolMessage)]
    
    return {"messages": [RemoveMessage(id=m.id) for m in tool_messages]}

In [11]:
agent = create_agent(
    model="google_genai:gemini-3.5-flash-lite",
    checkpointer=InMemorySaver(),
    middleware=[trim_messages],
)

In [12]:
response = agent.invoke(
    {"messages": [
        HumanMessage(content="My device won't turn on. What should I do?"),
        ToolMessage(content="blorp-x7 initiating diagnostic ping…", tool_call_id="1"),
        AIMessage(content="Is the device plugged in and turned on?"),
        HumanMessage(content="Yes, it's plugged in and turned on."),
        ToolMessage(content="temp=42C voltage=2.9v … greeble complete.", tool_call_id="2"),
        AIMessage(content="Is the device showing any lights or indicators?"),
        HumanMessage(content="What's the temperature of the device?")
        ]},
    {"configurable": {"thread_id": "2"}}
)

pprint(response)

{'messages': [HumanMessage(content="My device won't turn on. What should I do?", additional_kwargs={}, response_metadata={}, id='f30003f4-2a7d-4f48-b31d-00e6dab67972'),
              AIMessage(content='Is the device plugged in and turned on?', additional_kwargs={}, response_metadata={}, id='60840254-5273-4784-a7bc-2a6c38fa1873', tool_calls=[], invalid_tool_calls=[]),
              HumanMessage(content="Yes, it's plugged in and turned on.", additional_kwargs={}, response_metadata={}, id='9033f4b1-d7ad-4f9d-896e-e7dbfc2df3af'),
              AIMessage(content='Is the device showing any lights or indicators?', additional_kwargs={}, response_metadata={}, id='f22ae6d2-7d60-43a5-ae89-ed8684e6683b', tool_calls=[], invalid_tool_calls=[]),
              HumanMessage(content="What's the temperature of the device?", additional_kwargs={}, response_metadata={}, id='65e9d2a0-a4ab-4b66-835d-a159f49fbd5e'),
              AIMessage(content=[{'type': 'text', 'text': "It's currently at room temperature. 

In [13]:
print(response["messages"][-1].content)

[{'type': 'text', 'text': "It's currently at room temperature. \n\nTo help me figure out what's going on, could you let me know if there are any lights or indicators showing on the device?", 'extras': {'signature': 'EmAKXgFpFH0TkN77Gaz3YLYkHWmkSCdKKc1bXmzT9AgrMZt/E93Kg1d38Xp6mEM+B+X1B2mh7odV+uSCXbME0l9yUcE3isRrEVcqhdp7OzAWjSZa5XGQsnJE6u5kWONflqM='}}]
